# Hotel Booking DApp - Step-by-Step Demonstration

This notebook demonstrates the execution of the `HotelBooking` smart contract on a local **Ganache** EVM blockchain using **web3.py**.

* **Payer (Guest):** Initiates room reservations and transfers funds.
* **Recipient (Hotel):** Registers available room inventory and accepts check-ins.
* **Escrow Contract:** Holds booking funds securely in escrow until check-in, cancellation, or no-show conditions are triggered.

**Prerequisite:** Ensure a local Ganache node is running via `./scripts/start_ganache.sh` prior to executing this notebook.

## 1. Establish Connection to Ganache Network

In [ ]:
import sys
sys.path.insert(0, "../scripts")

from common import *
from web3.exceptions import ContractLogicError

w3 = connect()
acc = w3.eth.accounts
hotel, guest, guest2 = acc[HOTEL], acc[GUEST], acc[GUEST_2]

print("Connected:", w3.is_connected(), "| chain id:", w3.eth.chain_id, "| latest block:", w3.eth.block_number)
print("Hotel :", hotel, eth(w3, w3.eth.get_balance(hotel)))
print("Guest :", guest, eth(w3, w3.eth.get_balance(guest)))

## 2. Smart Contract Compilation and Deployment

In [ ]:
artifact = compile_contract()
contract, receipt = deploy(w3)
print("Contract address:", contract.address)
print("Deploy tx:", "0x" + receipt.transactionHash.hex(), "| block", receipt.blockNumber, "| gas", receipt.gasUsed)

## 3. Hotel Registration and Room Inventory Management

In [ ]:
send(w3, contract.functions.registerHotel("Liffey View Hotel"), hotel)
send(w3, contract.functions.addRoom("Double room, city view", w3.to_wei(0.05, "ether")), hotel)
send(w3, contract.functions.addRoom("Single room", w3.to_wei(0.03, "ether")), hotel)

for room_id in contract.functions.getHotelRooms(hotel).call():
    print(contract.functions.getRoom(room_id).call())

today = contract.functions.today().call()
print("today() =", today)

## 4. On-Chain Privacy Protection (Keccak-256 Hashing)
Personal identity information (such as full names and passport numbers) is never stored directly on-chain. Only a salted cryptographic hash is persisted, enabling off-chain identity verification while preserving data privacy.

In [ ]:
data_hash = guest_data_hash("Jane Doe", "P1234567", "random-salt-42")
print("Stored on-chain:", "0x" + data_hash.hex())
print("Demonstrating hash sensitivity (changing a single character):")
print("               ", "0x" + guest_data_hash("Jane Doe", "P1234568", "random-salt-42").hex())

## 5. Room Reservation and Escrow Deposit

In [ ]:
price = contract.functions.quotePrice(1, today, today + 2).call()
r = send(w3, contract.functions.bookRoom(1, today, today + 2, data_hash), guest, price)
ev = contract.events.RoomBooked().process_receipt(r)[0]["args"]
print("RoomBooked event:", dict(ev))
print("Escrow balance:", eth(w3, contract.functions.escrowBalance().call()))
print("Booking:", contract.functions.getBooking(1).call())

## 6. Validation Rules and Exception Handling Verification

In [ ]:
tests = [
    ("Double booking attempt", contract.functions.bookRoom(1, today + 1, today + 3, data_hash), guest2, 2 * price),
    ("Incorrect payment value", contract.functions.bookRoom(2, today, today + 1, data_hash), guest2, 1),
    ("Unauthorized cancellation attempt", contract.functions.cancelBooking(1), guest2, 0),
]
for name, fn, who, value in tests:
    try:
        fn.transact({"from": who, "value": value})
        print(name, "-> Transaction accepted (unexpected)")
    except ContractLogicError as e:
        print(name, "-> REVERTED:", e.message)

## 7. Guest Check-In Verification via ECDSA Signature
The guest cryptographically signs a check-in authorization off-chain using their private key. The contract utilizes `ecrecover` to recover the signer address and verify it against the booking record.

In [ ]:
from eth_account import Account
from eth_account.messages import encode_defunct

msg_hash = contract.functions.getCheckInMessageHash(1).call()
signature = sign_check_in(contract, 1, GUEST)
recovered = Account.recover_message(encode_defunct(primitive=msg_hash), signature=signature)
print("Signature :", "0x" + signature.hex())
print("Recovered :", recovered, "| matches guest:", recovered == guest)

# Verify that signatures generated by unauthorized accounts are rejected
try:
    contract.functions.checkIn(1, sign_check_in(contract, 1, GUEST_2)).transact({"from": hotel})
except ContractLogicError as e:
    print("Wrong signer ->", e.message)

send(w3, contract.functions.checkIn(1, signature), hotel)
print("Status after check-in:", contract.functions.getBooking(1).call()[8], "(2 = CheckedIn)")
print("Hotel can withdraw:", eth(w3, contract.functions.pendingWithdrawals(hotel).call()))

## 8. Cancellation Policy Enforcement (>48h Free, <48h Penalty Fee)

In [ ]:
p = contract.functions.quotePrice(2, today + 5, today + 6).call()
send(w3, contract.functions.bookRoom(2, today + 5, today + 6, data_hash), guest2, p)
r = send(w3, contract.functions.cancelBooking(2), guest2)
print("Early:", dict(contract.events.BookingCancelled().process_receipt(r)[0]["args"]))

p = contract.functions.quotePrice(2, today + 1, today + 2).call()
send(w3, contract.functions.bookRoom(2, today + 1, today + 2, data_hash), guest, p)
r = send(w3, contract.functions.cancelBooking(3), guest)
print("Late :", dict(contract.events.BookingCancelled().process_receipt(r)[0]["args"]))

## 9. Funds Settlement (Pull-Payment Pattern)

In [ ]:
for name, who in [("Hotel", hotel), ("Guest", guest), ("Guest 2", guest2)]:
    owed = contract.functions.pendingWithdrawals(who).call()
    if owed:
        send(w3, contract.functions.withdraw(), who)
    print(f"{name:8} withdrew {eth(w3, owed)}")
print("Escrow left:", eth(w3, contract.functions.escrowBalance().call()))

## 10. Immutable Shared Ledger Audit

In [ ]:
names = {hotel: "Hotel", guest: "Guest", guest2: "Guest2", acc[0]: "Deployer", contract.address: "Contract"}
start = receipt.blockNumber
for n in range(start, w3.eth.block_number + 1):
    block = w3.eth.get_block(n, full_transactions=True)
    for tx in block.transactions:
        fn = "deploy" if tx["to"] is None else contract.decode_function_input(tx["input"])[0].fn_name
        print(f"block {n:3} | {names.get(tx['from'], tx['from'][:8]):8} -> {fn:14} | value {eth(w3, tx['value'])}")